**Nombre de Proyecto:** Proyecto EdTech Learning Analytics Parte 2

**Numero de grupo:** 2

**Integrante a cargo:** Gabriela Michelle Garcia Perez

**Carnet:** 038425

**Fecha de creacion:** 15/09/2026

**Version de libro:** 1.3

In [1]:
# ==========================================
# PROYECTO EDTECH LEARNING ANALYTICS - PARTE 3
#ARQUITECTURA DE BIG DATA CON SPARK
# ==========================================

%pip install pyspark
print("PySpark instalado ----------------")

PySpark instalado ----------------


In [2]:
#IMPORTAMOS LIBRERIAS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd
import shutil, os

print("Librerias importadas --------- ")

Librerias importadas --------- 


In [3]:
#CREAR SESION DE SPARK
spark = SparkSession.builder \
  .master("local[*]")\
  .appName("EdTech_LearningAnalytics_Guia03_Gabriela_Garcia")\
  .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Paralelismo: {spark.sparkContext.defaultParallelism}")

Spark version: 4.0.4
Paralelismo: 2


In [4]:
#DEFINIR RUTAS
BASE_DIR = Path("/content")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_ANALYTICS = BASE_DIR/"datos"/"analiticos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)

NOMBRE_CSV = "dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv"

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos analiticos: {DATA_ANALYTICS}")


Datos procesados: /content/datos/procesados
Datos analiticos: /content/datos/analiticos


In [5]:
#SUBIR EL CSV A COLAB Y COLOCARLO EN LA CARPETA DE PROCESADOS
import shutil
from google.colab import files

NOMBRE_CSV = "dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv"
ruta_entrada = DATA_PROCESSED/NOMBRE_CSV

if not ruta_entrada.exists():
    subidos = files.upload()          # selecciona el CSV cuando te aparezca el botón
    nombre_subido = list(subidos.keys())[0]
    shutil.move(nombre_subido, ruta_entrada)
    print("CSV subido y movido --------")
else:
    print("El CSV ya estaba en la carpeta --------")

print(f"Archivo listo en: {ruta_entrada}")

Saving dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv to dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
CSV subido y movido --------
Archivo listo en: /content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv


In [6]:
#CARGAR DATASET LIMPIO EN SPARK

print(f"Leyendo: {ruta_entrada}")

# 'historial_cualitativo' guarda texto tipo JSON con comas dentro,
# por eso se necesita quote/escape/multiLine para que Spark no corte mal las filas
df = spark.read\
  .option("header", True)\
  .option("inferSchema", True)\
  .option("quote", '"')\
  .option("escape", '"')\
  .option("multiLine", True)\
  .csv(str(ruta_entrada))

print("Dataset cargado en Spark ------")
print(f"Filas: {df.count()}")
print(f"Columnas: {len(df.columns)}")

Leyendo: /content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
Dataset cargado en Spark ------
Filas: 1000
Columnas: 43


In [7]:
#VER EL ESQUEMA - MOSTRAR LA ESTRUCTURA DEL DATASET
print("Esquema de datos: -------------")
df.printSchema() #Con pandas es con .info

print("\nPrimeras filas: ")
df.select("carnet", "materia", "nota_teoria", "nota_laboratorio",
          "nota_final", "nivel_riesgo", "probabilidad_desercion").show(5, truncate=False)

Esquema de datos: -------------
root
 |-- carnet: string (nullable = true)
 |-- nombres: string (nullable = true)
 |-- apellidos: string (nullable = true)
 |-- fecha_nacimiento: date (nullable = true)
 |-- edad: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- facultad: string (nullable = true)
 |-- carrera: string (nullable = true)
 |-- anio_ingreso: integer (nullable = true)
 |-- ciclo_actual: integer (nullable = true)
 |-- turno: string (nullable = true)
 |-- municipio_residencia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- telefono: integer (nullable = true)
 |-- correo_institucional: string (nullable = true)
 |-- acceso_internet_casa: string (nullable = true)
 |-- trabaja: string (nullable = true)
 |-- horas_trabajo_semana: double (nullable = true)
 |-- beca: string (nullable = true)
 |-- ciclo: integer (nullable = true)
 |-- materia: string (nullable = true)
 |-- nota_teoria: double (nullable = true)
 |-- nota_laboratorio: double (nullable 

In [8]:
#PARTICIONES CON SPARK
print(f"\nParticiones iniciales: {df.rdd.getNumPartitions()}")



Particiones iniciales: 1


In [9]:
#FILTRAR REGISTROS VALIDOS
df_validos = df.filter(
    (F.col("nota_final") > 0) &
    F.col("materia").isNotNull() &
    F.col("nivel_riesgo").isNotNull()
)

print(f"Registros despues de filtrar: {df_validos.count()}")
print(f"Registros eliminados: {df.count() - df_validos.count()}")


Registros despues de filtrar: 1000
Registros eliminados: 0


In [10]:
#AQUI PODEMOS VER QUE LA PARTE 2 ESTA CORRECTAMENTE EJECUTADA, YA QUE LOS DATOS YA VENIAN LIMPIOS

In [11]:
#CREAR COLUMNA DERIVADA -- MARCAR ESTUDIANTES EN RIESGO ALTO

df_enriquecido = df_validos.withColumn(
    "en_riesgo_alto",
    F.when(F.col("nivel_riesgo") == "Alto", True).otherwise(False)
)

print("Columna 'en_riesgo_alto' creada --------------")
df_enriquecido.select("carnet", "materia", "nivel_riesgo", "en_riesgo_alto").show(5)

Columna 'en_riesgo_alto' creada --------------
+-------+--------------+------------+--------------+
| carnet|       materia|nivel_riesgo|en_riesgo_alto|
+-------+--------------+------------+--------------+
|RR19001|        Ingles|        Bajo|         false|
|RF24002|   Estadistica|        Alto|          true|
|FG23003|      Fisica I|        Bajo|         false|
|RL21004|        Ingles|       Medio|         false|
|PM21005|Programacion I|       Medio|         false|
+-------+--------------+------------+--------------+
only showing top 5 rows


In [12]:
#AGREGAR POR MATERIA

resumen_materia = df_enriquecido.groupBy("materia").agg(
    F.round(F.avg("nota_final"), 2).alias("promedio_nota_final"),
    F.count("*").alias("estudiantes"),
    F.sum(F.when(F.col("en_riesgo_alto"), 1).otherwise(0)).alias("estudiantes_riesgo_alto")
).orderBy(F.desc("promedio_nota_final"))

print("Promedio academico por curso: -------")
resumen_materia.show(truncate=False)

Promedio academico por curso: -------
+--------------+-------------------+-----------+-----------------------+
|materia       |promedio_nota_final|estudiantes|estudiantes_riesgo_alto|
+--------------+-------------------+-----------+-----------------------+
|Estadistica   |7.23               |169        |44                     |
|Programacion I|7.08               |178        |53                     |
|Base de Datos |7.04               |164        |52                     |
|Fisica I      |7.03               |154        |47                     |
|Matematica I  |7.02               |167        |50                     |
|Ingles        |6.97               |168        |55                     |
+--------------+-------------------+-----------+-----------------------+



In [13]:
#CREAR VISTA TEMPORAL
df_enriquecido.createOrReplaceTempView("rendimiento_academico")
print("Vista temporal 'rendimiento_academico' creada")


Vista temporal 'rendimiento_academico' creada


In [14]:
#CREAR CONSULTA SQL - PROMEDIO Y RIESGO POR MATERIA

consulta = """
  SELECT
    materia,
    COUNT(*) AS estudiantes,
    ROUND(AVG(nota_final), 2) AS promedio_nota_final,
    SUM(CASE WHEN nivel_riesgo = 'Alto' THEN 1 ELSE 0 END) AS estudiantes_riesgo_alto,
    ROUND(100 * SUM(CASE WHEN nivel_riesgo = 'Alto' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_riesgo_alto
  FROM rendimiento_academico
  GROUP BY materia
  ORDER BY promedio_nota_final DESC
"""

resultado_sql = spark.sql(consulta)
print("Consulta SQL - PROMEDIO Y RIESGO POR MATERIA")
resultado_sql.show(truncate=False)

Consulta SQL - PROMEDIO Y RIESGO POR MATERIA
+--------------+-----------+-------------------+-----------------------+---------------+
|materia       |estudiantes|promedio_nota_final|estudiantes_riesgo_alto|pct_riesgo_alto|
+--------------+-----------+-------------------+-----------------------+---------------+
|Estadistica   |169        |7.23               |44                     |26.0           |
|Programacion I|178        |7.08               |53                     |29.8           |
|Base de Datos |164        |7.04               |52                     |31.7           |
|Fisica I      |154        |7.03               |47                     |30.5           |
|Matematica I  |167        |7.02               |50                     |29.9           |
|Ingles        |168        |6.97               |55                     |32.7           |
+--------------+-----------+-------------------+-----------------------+---------------+



In [15]:
#VER PLAN DE EJECUCION
print("\nPLAN DE EJECUCION FORMATEADO: ------------")
resultado_sql.explain("formatted")


PLAN DE EJECUCION FORMATEADO: ------------
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [3]: [materia#20, nota_final#23, nivel_riesgo#30]
Batched: false
Location: InMemoryFileIndex [file:/content/datos/procesados/dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv]
PushedFilters: [IsNotNull(nota_final), GreaterThan(nota_final,0.0), IsNotNull(materia), IsNotNull(nivel_riesgo)]
ReadSchema: struct<materia:string,nota_final:double,nivel_riesgo:string>

(2) Filter
Input [3]: [materia#20, nota_final#23, nivel_riesgo#30]
Condition : (((isnotnull(nota_final#23) AND (nota_final#23 > 0.0)) AND isnotnull(materia#20)) AND isnotnull(nivel_riesgo#30))

(3) HashAggregate
Input [3]: [materia#20, nota_final#23, nivel_riesgo#30]
Keys [1]: [materia#20]
Functions [3]: [partial_count(1), partial_avg(nota_fi

In [16]:
#INSPECCIONAR PARTICIONES ACTUALES
print(f"Particiones actuales: {df_enriquecido.rdd.getNumPartitions()}")


Particiones actuales: 1


In [17]:
#REPARTICIONAR EN 4
df_particionado = df_enriquecido.repartition(4)
print(f"Nuevas particiones: {df_particionado.rdd.getNumPartitions()}")

Nuevas particiones: 4


In [18]:
#GUARDAR EN PARQUET
ruta_parquet = DATA_ANALYTICS/"rendimiento_academico_parquet"

df_particionado.write \
  .mode("overwrite")\
  .parquet(str(ruta_parquet))

print(f"Parquet guardado en: {ruta_parquet}")

Parquet guardado en: /content/datos/analiticos/rendimiento_academico_parquet


In [19]:
#LEER PARQUET
df_leido = spark.read.parquet(str(ruta_parquet))

print(f"Filas leidas: {df_leido.count()}")
print("Primeras filas: ------------------")
df_leido.select("carnet", "materia", "nota_final", "nivel_riesgo", "en_riesgo_alto").show(5, truncate=False)

Filas leidas: 1000
Primeras filas: ------------------
+-------+-------------+----------+------------+--------------+
|carnet |materia      |nota_final|nivel_riesgo|en_riesgo_alto|
+-------+-------------+----------+------------+--------------+
|HM21967|Fisica I     |7.38      |Medio       |false         |
|FM21476|Matematica I |6.8       |Alto        |true          |
|LG23888|Ingles       |7.12      |Alto        |true          |
|RG21441|Base de Datos|6.76      |Alto        |true          |
|VÁ24327|Ingles       |8.36      |Bajo        |false         |
+-------+-------------+----------+------------+--------------+
only showing top 5 rows


In [20]:
#AGREGACIONES CON PANDAS

df_pd = pd.read_csv(ruta_entrada)
df_pd = df_pd[(df_pd["nota_final"] > 0) & df_pd["materia"].notna() & df_pd["nivel_riesgo"].notna()]

pd_materia = df_pd.groupby("materia")["nota_final"].mean().round(2).reset_index()
pd_materia.columns = ["materia", "promedio_pandas"]

print("Pandas - promedio de nota final por materia")
print(pd_materia)

Pandas - promedio de nota final por materia
          materia  promedio_pandas
0   Base de Datos             7.04
1     Estadistica             7.23
2        Fisica I             7.03
3          Ingles             6.97
4    Matematica I             7.02
5  Programacion I             7.08


In [21]:
#RESULTADO SPARK (AGREGACIONES) PARA COMPARAR CON PANDAS

sp_materia = resumen_materia.select("materia", "promedio_nota_final")\
                            .orderBy("materia").toPandas()
sp_materia.columns = ["materia", "promedio_spark"]

print("\nSPARK - PROMEDIO DE NOTA FINAL POR MATERIA")
print(sp_materia)


SPARK - PROMEDIO DE NOTA FINAL POR MATERIA
          materia  promedio_spark
0   Base de Datos            7.04
1     Estadistica            7.23
2        Fisica I            7.03
3          Ingles            6.97
4    Matematica I            7.02
5  Programacion I            7.08


In [22]:
#COMPARAR RESULTADOS
print("\nComparacion Pandas vs Spark")

comparacion = pd.merge(pd_materia, sp_materia, on="materia")
comparacion["diferencia"] = (comparacion["promedio_pandas"] - comparacion["promedio_spark"]).abs().round(6)
print(comparacion)

if comparacion["diferencia"].max() < 0.001:
    print("\nLos resultados son consistentes entre Pandas y Spark --------- ")
else:
    print("\nHay diferencias, revisar filtros aplicados")


Comparacion Pandas vs Spark
          materia  promedio_pandas  promedio_spark  diferencia
0   Base de Datos             7.04            7.04         0.0
1     Estadistica             7.23            7.23         0.0
2        Fisica I             7.03            7.03         0.0
3          Ingles             6.97            6.97         0.0
4    Matematica I             7.02            7.02         0.0
5  Programacion I             7.08            7.08         0.0

Los resultados son consistentes entre Pandas y Spark --------- 


In [23]:
#REPORTE DE ARQUITECTURA
print("\nREPORTE DE ARQUITECTURA - PROYECTO FINAL EDTECH")
print("=" * 55)
print(f"Versión de Spark: {spark.version}")
print(f"Dataset de entrada: {NOMBRE_CSV}")
print(f"Filas procesadas: {df_enriquecido.count()}")
print(f"Columnas originales: {len(df.columns)}")
print(f"Columnas finales: {len(df_particionado.columns)}")
print(f"Particiones iniciales: 1")
print(f"Particiones finales: {df_particionado.rdd.getNumPartitions()}")
print(f"Vista temporal SQL: rendimiento_academico")
print(f"Salida Parquet: {ruta_parquet}")
print(f"Materia con mejor promedio: Estadistica (7.23)")
print(f"Materia con mayor % de riesgo alto: Ingles (32.7%)")
print("=" * 55)


REPORTE DE ARQUITECTURA - PROYECTO FINAL EDTECH
Versión de Spark: 4.0.4
Dataset de entrada: dataset_limpio_Proyecto_EdTech_Learning_Analytics.csv
Filas procesadas: 1000
Columnas originales: 43
Columnas finales: 44
Particiones iniciales: 1
Particiones finales: 4
Vista temporal SQL: rendimiento_academico
Salida Parquet: /content/datos/analiticos/rendimiento_academico_parquet
Materia con mejor promedio: Estadistica (7.23)
Materia con mayor % de riesgo alto: Ingles (32.7%)


In [24]:
#CERRAR SESION
spark.stop()
print("Sesion de Spark finalizada!!!!!")

Sesion de Spark finalizada!!!!!


In [26]:
#DESCARGAR TODOS LOS ARCHIVOS GENERADOS
!zip -r /content/TeceraParte_semana3.zip /content/datos/

from google.colab import files
files.download("/content/TeceraParte_semana3.zip")

print("Todos los archivos descargados!")

  adding: content/datos/ (stored 0%)
  adding: content/datos/analiticos/ (stored 0%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/ (stored 0%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/part-00003-ba5353ea-c249-4efa-9dfa-89a952741c72-c000.snappy.parquet (deflated 33%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/._SUCCESS.crc (stored 0%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/part-00001-ba5353ea-c249-4efa-9dfa-89a952741c72-c000.snappy.parquet (deflated 32%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/part-00000-ba5353ea-c249-4efa-9dfa-89a952741c72-c000.snappy.parquet (deflated 33%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/part-00002-ba5353ea-c249-4efa-9dfa-89a952741c72-c000.snappy.parquet (deflated 33%)
  adding: content/datos/analiticos/rendimiento_academico_parquet/.part-00002-ba5353ea-c249-4efa-9dfa-89a952741c72-c000.snappy.parquet.crc (stored 0%)
  ad

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Todos los archivos descargados!
